# 1. Read and check the dataset

This notebook uses the settings from setup. The runner executes it first. Training uses all available training rows after validation is reserved. Original test rows stay held out.

## Imports and required earlier steps

Choose the **Whisper (project .venv)** kernel. This cell imports the libraries and loads missing settings/prerequisites. Training stays disabled until you explicitly enable it.

In [ ]:
from pathlib import Path
import json
import hashlib
import random
import unicodedata
import numpy as np
import pandas as pd
import soundfile as sf
import torch

# Load settings when this notebook starts in a fresh kernel.
if "DATA_DIR" not in globals():
    root = Path.cwd().resolve()
    while not (root / "pyproject.toml").is_file():
        if root.parent == root:
            raise FileNotFoundError("Start the kernel inside the AI & ML project.")
        root = root.parent
    setup = json.loads((root / "notebooks/whisper/setup.ipynb").read_text())
    for cell in setup["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), globals())


## Read the two CSV files

Each row contains a relative audio path, its transcript, and duration. Copies are kept so we can check that test membership remains unchanged.

In [ ]:
original_train = pd.read_csv(DATA_DIR / "train.csv", keep_default_na=False,
                             dtype={"audio_path": str, "text": str})
original_test = pd.read_csv(DATA_DIR / "test.csv", keep_default_na=False,
                            dtype={"audio_path": str, "text": str})
assert len(original_train) > 1, "Training needs at least two recordings."
assert len(original_test) > 0, "The test set is empty."
print("Original train:", len(original_train))
print("Original test:", len(original_test))

## Two small helpers

`text_group` gives repeated transcripts the same key. It changes only duplicate-detection keys, never the actual transcript.

`read_audio` checks the prepared audio instead of silently changing or cutting it.

In [ ]:
def text_group(text):
    text = unicodedata.normalize("NFC", text).lower()
    text = "".join(letter if letter.isalnum() or letter.isspace() else " " for letter in text)
    return " ".join(text.split())


def read_audio(path):
    audio, sample_rate = sf.read(path, dtype="float32")
    if sample_rate != SAMPLE_RATE or audio.ndim != 1:
        raise ValueError(f"Expected mono {SAMPLE_RATE} Hz audio: {path}")
    if not 0 < len(audio) <= SAMPLE_RATE * 30:
        raise ValueError(f"Audio must be nonempty and at most 30 seconds: {path}")
    if not np.isfinite(audio).all():
        raise ValueError(f"Invalid audio samples: {path}")
    return audio

## Check every recording

We check paths, audio, and durations. This prepared corpus has no exact audio duplicates; finding one stops training for review. Repeated transcripts are allowed and kept together when splitting.

In [ ]:
audio_hashes = set()
for table in [original_train, original_test]:
    table["group"] = table["text"].map(text_group)
    assert table["group"].ne("").all(), "An empty transcript was found."
    assert not table["audio_path"].duplicated().any(), "Duplicate audio paths were found."

    for row in table.to_dict("records"):
        path = (DATA_DIR / row["audio_path"]).resolve()
        if Path(row["audio_path"]).is_absolute() or not path.is_relative_to(DATA_DIR):
            raise ValueError(f"Audio path must stay inside the dataset: {path}")
        audio = read_audio(path)
        duration = float(row["duration"])
        if not np.isfinite(duration) or abs(duration - len(audio) / SAMPLE_RATE) > 0.02:
            raise ValueError(f"Incorrect duration: {path}")
        audio_hash = hashlib.sha256(audio.tobytes()).hexdigest()
        if audio_hash in audio_hashes:
            raise ValueError(f"Duplicate audio found; review the dataset before training: {path}")
        audio_hashes.add(audio_hash)

assert set(original_train["group"]).isdisjoint(original_test["group"]), "Train/test transcript overlap."
print("All recordings passed the audio and transcript checks.")

## Reserve about 10% of train for validation

Shuffle transcript groups with a fixed seed, then collect whole groups until we reach the validation target. The remaining rows are training rows.

In [ ]:
groups = original_train["group"].drop_duplicates().tolist()
assert len(groups) >= 2, "We need at least two different transcript groups."
random.Random(SEED).shuffle(groups)

validation_groups = []
validation_count = 0
validation_target = max(1, int(len(original_train) * VALIDATION_FRACTION))
for group in groups[:-1]:  # Leave at least one group for training.
    validation_groups.append(group)
    validation_count += int(original_train["group"].eq(group).sum())
    if validation_count >= validation_target:
        break

is_validation = original_train["group"].isin(validation_groups)
train_df = original_train[~is_validation].reset_index(drop=True)
validation_df = original_train[is_validation].reset_index(drop=True)
test_df = original_test.copy().reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))
print("Example transcript:", train_df.iloc[0]["text"])